# Python data cleaning - TranxMex Analisis y Optimizacion Finanaciera

---

**Fecha de creación:** Julio 2026

**Autor:** David Alexander Pupo González

### Tabla de contenido

---

1) Importar librerías y definir rutas
2) Cargar el CSV bruto
3) Normalizar nombres de columnas
4) Validar columnas obligatorias
5) Limpiar la columna de [ fechas ]
6) Limpiar la columna de [ importes ]
7) Normalizar la columna de [ moneda ]
8) Normalizar la columna de [ estado ]
9) Normalizar la columna de [ categoría ]
10) Limpiar columnas de texto
11) Tratar nulos útiles en [ cliente_proveedor ]
12) Eliminar duplicados
13) Filtrar registros válidos [ importe_limpio ]
14) Crear columnas temporales [ anio ] [ mes ] [ nombre_mes ]
15) Seleccionar columnas finales
16) Guardar archivos finales
17) Se crea una tabla con métricas del proceso.


### Propósito del proyecto

---

Este proyecto busca limpiar el set de datos de la empresa TranxMex para ser usado después en análisis de datos, describiendo los pasos usados uno por uno para este ejemplo, aunque se puede extrapolar a casi cualquier otra situación.


### 1: Importar librerías y definir rutas

Aquí se cargan las bibliotecas necesarias y se definen la ruta de entrada y la carpeta de salida.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

inp = Path('input/transacciones_brutas.csv')
out = Path('output')
out.mkdir(exist_ok=True)


Descripción:
* pandas para manipular tablas.
* numpy para valores nulos numéricos.
* Path para manejar rutas de archivos.
* out.mkdir(exist_ok=True) crea la carpeta de salida si no existe.


### 2: Cargar el CSV bruto
En este paso se lee el archivo original.


In [2]:
df = pd.read_csv(inp, sep=',')


Descripción:

Carga el dataset fuente transacciones_brutas.csv en un DataFrame para empezar la limpieza.


### 3: Normalizar nombres de columnas
Primero se limpian los nombres de columnas, porque eso facilita todo lo que viene después.


In [3]:
print('Columnas originales:', df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('\nColumnas limpias:', df.columns.tolist())


Columnas originales: ['transaction_id', 'fecha', 'empresa', 'cliente_proveedor', 'categoria', 'subcategoria', 'tipo_movimiento', 'importe', 'moneda', 'canal_pago', 'estado', 'centro_coste', 'descripcion', 'referencia_externa', 'pais']

Columnas limpias: ['transaction_id', 'fecha', 'empresa', 'cliente_proveedor', 'categoria', 'subcategoria', 'tipo_movimiento', 'importe', 'moneda', 'canal_pago', 'estado', 'centro_coste', 'descripcion', 'referencia_externa', 'pais']


Descripción:
* Quita espacios sobrantes.
* Convierte todo a minúsculas.
* Reemplaza espacios por guiones bajos.
Esto evita errores cuando se referencian columnas por nombre.


### 4: Validar columnas obligatorias
Antes de limpiar valores, se verifica que existan las columnas necesarias.


In [4]:
required = ['fecha', 'importe', 'moneda', 'estado', 'categoria']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Faltan columnas requeridas: {missing}. Disponibles: {df.columns.tolist()}')


Descripción:

Si falta una columna crítica, el script se detiene. Esto evita que el proceso siga con un archivo incompleto o mal estructurado.


### 5: Limpiar la columna de [ HECHAS ]
Ahora se aplica una función específica para convertir fechas en distintos formatos a un formato estándar.


In [5]:
def parse_fecha(x):
    if pd.isna(x):
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d', '%Y-%m-%dT%H:%M:%S'):
        try:
            return pd.to_datetime(x, format=fmt)
        except Exception:
            pass
    return pd.to_datetime(x, errors='coerce')

df['fecha'] = df['fecha'].apply(parse_fecha)


Descripción:
* Detecta nulos.
* Prueba varios formatos de fecha.
* Si ninguno funciona, intenta convertir automáticamente.
* Si no se puede, deja NaT.


### 6: Limpiar la columna de [ IMPORTES ]
Luego se limpia el importe para poder tratarlo como número real.


In [6]:
def clean_importe(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    for token in ['EUR', '€', 'usd', 'USD', 'Sterling', 'GBP']:
        x = x.replace(token, '')
    x = x.replace(' ', '').replace(',', '.')
    return pd.to_numeric(x, errors='coerce')

df['importe_limpio'] = df['importe'].apply(clean_importe)


Descripción:
* Convierte el valor a texto.
* Elimina símbolos o palabras de moneda.
* Cambia coma decimal por punto.
* Convierte a número.
* Si falla, deja NaN.


### 7: Normalizar la columna de [ MONEDA ]
Aquí se estandarizan las distintas formas de escribir una moneda.


In [7]:
map_moneda = {
    '€': 'EUR',
    'eur': 'EUR',
    'EUR': 'EUR',
    'usd': 'USD',
    'USD': 'USD',
    'Sterling': 'GBP',
    'sterling': 'GBP',
    'GBP': 'GBP'
}
df['moneda'] = df['moneda'].replace(map_moneda).astype(str).str.upper().str.strip()


Descripción:

Convierte todas las variantes a códigos estándar: EUR, USD, GBP.


### 8: Normalizar la columna de [ ESTADO ]
Se unifican los valores posibles de estado.


In [8]:
map_estado = {
    'PAID': 'Pagado',
    'pagado': 'Pagado',
    'Pagado': 'Pagado',
    'pendiente': 'Pendiente',
    'Pendiente': 'Pendiente',
    'rechazado': 'Rechazado',
    'Rechazado': 'Rechazado',
    'Cancelado': 'Cancelado',
    'cancelado': 'Cancelado'
}
df['estado'] = df['estado'].replace(map_estado).astype(str).str.strip().str.title()


Descripción:

Con esto todos los estados quedan escritos de forma homogénea.


### 9: Normalizar la columna de [ CATEGORÍA ]

Se corrigen variaciones de escritura en las categorías.


In [9]:
map_categoria = {
    'venta': 'Ventas',
    'Ventas': 'Ventas',
    'mkt': 'Marketing',
    'Marketing': 'Marketing',
    'Consultoria': 'Consultoría',
    'Consultoria ': 'Consultoría',
    'Consultoría': 'Consultoría'
}
df['categoria'] = df['categoria'].replace(map_categoria).astype(str).str.strip().str.title()


Descripción:

Unifica nombres de categorías para que no aparezcan duplicadas por diferencias de escritura.


### 10: Limpiar columnas de texto
Las columnas textuales se limpian con una regla común.


In [10]:
text_cols = ['empresa', 'cliente_proveedor', 'canal_pago', 'centro_coste', 'descripcion', 'pais', 'subcategoria', 'tipo_movimiento']
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
        

Descripción:
* Convierte a texto.
* Quita espacios externos.
* Reduce múltiples espacios internos a uno solo.


### 11: Tratar nulos útiles en [ CLIENTE_PROVEEDOR ]
Se preserva la información faltante como una categoría explícita.


In [11]:
if 'cliente_proveedor' in df.columns:
    df['cliente_proveedor'] = (
        df['cliente_proveedor']
        .astype('string')
        .str.strip()
        .replace({'None': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, '': pd.NA})
        .fillna('DESCONOCIDO')
    )


Descripción:

En vez de borrar filas, se etiqueta el dato faltante como DESCONOCIDO, lo cual sirve para análisis posteriores.


### 12: Eliminar duplicados
Se eliminan filas repetidas.


In [12]:
before_dupes = len(df)
df = df.drop_duplicates().copy()
after_dupes = len(df)


Descripción:
* Cuenta filas antes.
* Elimina duplicados.
* Cuenta filas después para medir cuántos se borraron.


### 13: Filtrar registros válidos [ IMPORTE_LIMPIO ]
Se dejan solo filas útiles para análisis.


In [13]:
df['importe_limpio'] = df['importe_limpio'].abs()
df = df[df['fecha'].notna()].copy()
df = df[df['importe_limpio'].notna()].copy()
df = df[df['importe_limpio'] > 0].copy()


Descripción:
* Convierte el importe a valor absoluto.
* Elimina filas sin fecha válida.
* Elimina filas con importe no convertible.
* Elimina importes cero.


### 14: Crear columnas temporales [ ANIO ] [ MES ] [ NOMBRE_MES ]
Se generan columnas auxiliares para análisis temporal.


In [14]:
mes_map = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
    7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month
df['nombre_mes'] = df['mes'].map(mes_map)


Descripción:

Permite analizar el dataset por año y por mes, tanto numérico como con nombre legible.


### 15: Seleccionar columnas finales
Se define la estructura final del dataset limpio.


In [15]:
final_cols = [
    'transaction_id', 'fecha', 'anio', 'mes', 'nombre_mes',
    'empresa', 'cliente_proveedor', 'categoria', 'subcategoria',
    'tipo_movimiento', 'importe_limpio', 'moneda', 'canal_pago',
    'estado', 'centro_coste', 'descripcion', 'referencia_externa', 'pais'
]
final_cols = [c for c in final_cols if c in df.columns]
df_final   = df[final_cols].copy()


Descripción:

Se deja solo la información útil, ordenada y lista para reporting.


### 16: Guardar archivo final
Se exporta el resultado limpio y un resumen de limpieza.


In [16]:
clean_file  = out / 'transacciones_limpias.csv'
df_final.to_csv(clean_file, index=False, encoding='utf-8-sig')


Descripción:

Guarda el dataset final limpio para usarlo en análisis o visualización.


### 17: Se crea y guarda tabla con métricas del proceso.

In [17]:
resumen = pd.DataFrame([
    ['filas_iniciales', len(pd.read_csv(inp, sep=','))],
    ['filas_sin_duplicados', after_dupes],
    ['duplicados_eliminados', before_dupes - after_dupes],
    ['filas_finales', len(df_final)],
    ['columnas_finales', df_final.shape[1]],
    ['nulos_fecha', int(df['fecha'].isna().sum())],
    ['nulos_importe_limpio', int(df['importe_limpio'].isna().sum())],
], columns=['metric', 'value'])

report_file = out / 'reporte_limpieza.csv'
resumen.to_csv(report_file, index=False, encoding='utf-8-sig')

print('\n✅ Limpieza completada')
print('Archivo limpio:', clean_file)
print('Reporte       :', report_file)
print('\nResumen:')
print(resumen.to_string(index=False))
print('\nPrimeras filas:')
print(df_final.head().to_string(index=False))



✅ Limpieza completada
Archivo limpio: output\transacciones_limpias.csv
Reporte       : output\reporte_limpieza.csv

Resumen:
               metric  value
      filas_iniciales   1040
 filas_sin_duplicados   1002
duplicados_eliminados     38
        filas_finales   1002
     columnas_finales     18
          nulos_fecha      0
 nulos_importe_limpio      0

Primeras filas:
transaction_id      fecha  anio  mes nombre_mes            empresa            cliente_proveedor   categoria subcategoria tipo_movimiento  importe_limpio moneda canal_pago estado centro_coste                                                  descripcion referencia_externa   pais
    TRX-100000 2026-01-29  2026    1      Enero  Nordic Capital SA   Banca Privada OLMJ S.L.N.E   Servicios      Soporte           Gasto         3997.05    EUR      Bizum Pagado    Dirección                                 Función sobre sí lado clase.          REF-81482 España
    TRX-100001 2024-01-31  2024    1      Enero  Nordic Capital SA  S

Descripción:

Este resumen permite explicar rápidamente qué pasó durante la limpieza: cuántas filas había, cuántas quedaron y qué problemas se detectaron.


### Orden conceptual correcto
Si lo quieres entender como proceso, el flujo correcto sería este:
1. Preparar entorno y rutas.
2. Cargar datos.
3. Limpiar nombres de columnas.
4. Validar columnas obligatorias.
5. Limpiar cada tipo de dato según su naturaleza:
* fechas,
* importes,
* monedas,
* estados,
* categorías,
* textos,
* nulos.
6. Eliminar duplicados.
7. Filtrar registros inválidos.
8. Crear columnas derivadas.
9. Seleccionar columnas finales.
10. Exportar resultados.


### Funciones usadas en la limpieza

```Función / método```  **Descripción**


```Path()```	Crea rutas de archivos y carpetas de forma segura y portable. Se usa para apuntar al CSV original y a la carpeta output.

```.mkdir(exist_ok=True)```	Crea la carpeta de salida si no existe; si ya existe, no da error.

```pd.read_csv()```	Lee el archivo CSV bruto y lo carga como DataFrame para poder limpiarlo.

```.str.strip()```	Elimina espacios al inicio y al final de los nombres de columnas o valores de texto.

```.str.lower()```	Convierte texto a minúsculas para normalizar nombres de columnas.

```.str.replace(' ', '_')```	Sustituye espacios por guiones bajos en los nombres de columnas.

```raise ValueError()```	Detiene el script si faltan columnas obligatorias, evitando continuar con un dataset incompleto.

```pd.isna()```	Detecta valores vacíos o nulos antes de limpiarlos.

```pd.NaT```	Representa una fecha nula cuando un valor no se puede interpretar como fecha.

```pd.to_datetime()```	Convierte texto o fechas en formato string a tipo fecha real de pandas.

```.apply(parse_fecha)```	Aplica la función de parseo de fechas a toda la columna fecha.

```np.nan```	Representa un valor numérico ausente al limpiar importes.

```str(x).strip()```	Convierte el importe a texto y quita espacios para poder limpiarlo mejor.

```.replace(token, '')```	Elimina símbolos o textos de moneda como EUR, USD, €, GBP.

```.replace(',', '.')```	Normaliza el separador decimal para que pandas pueda convertir el valor a número.

```pd.to_numeric(..., errors='coerce')```	Convierte el texto limpio a número; si falla, devuelve NaN.

```.replace(map_moneda)```	Traduce variantes de moneda a códigos estándar como EUR, USD o GBP.

```.astype(str)```	Convierte la columna a texto para poder aplicar normalización uniforme.

```.str.upper()```	Pone el texto en mayúsculas para unificar valores como eur y EUR.

```.str.title()```	Capitaliza valores de texto, útil para estados como Pagado, Pendiente, etc.

```.replace(map_estado)```	Normaliza distintos formatos de estado a una sola versión estándar.

```.replace(map_categoria)```	Unifica categorías escritas de forma distinta en un mismo valor estándar.

```for c in text_cols:```	Recorre columnas de texto para limpiarlas de manera uniforme.

```if c in df.columns:```	Verifica que la columna exista antes de intentar limpiarla.

```.str.replace(r'\s+', ' ', regex=True)```	Reduce múltiples espacios internos a un solo espacio.

```.replace('None', np.nan)```	Convierte el texto literal "None" en valor nulo real.

```.fillna('DESCONOCIDO')```	Rellena valores faltantes de cliente/proveedor con una etiqueta útil para análisis.

```.drop_duplicates()```	Elimina filas duplicadas dejando una sola copia de cada registro.

```.copy()```	Crea una copia limpia del DataFrame para evitar problemas de referencia.

```.abs()```	Convierte el importe a valor absoluto para evitar signos negativos en reportes financieros.

```.notna()```	Filtra filas con fecha válida y elimina registros no utilizables para análisis temporal.

```df[df['importe_limpio'] > 0]```	Elimina transacciones de importe 0, que no aportan valor analítico.

```.dt.year```	Extrae el año desde la columna fecha.

```.dt.month```	Extrae el número del mes desde la fecha.

```.map(mes_map)```	Convierte el número de mes en nombre del mes en español.

```df[final_cols]```	Selecciona solo las columnas finales deseadas para el dataset limpio.

```.to_csv()```	Guarda el CSV limpio y el resumen de limpieza en la carpeta de salida.

```len(df)```	Cuenta filas para medir volumen antes y después de la limpieza.

```df.shape[^1]```	Cuenta el número de columnas finales.

```.isna().sum()```	Cuenta cuántos nulos quedan en una columna.

